<a href="https://colab.research.google.com/github/DrKroll/llm-humanidades-salamanca/blob/main/notebooks/extras/E2_calderon_estilo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# E2 · (Extra) Ajuste de **estilo**: escribir «como Calderón»

**Objetivo:** ver la otra cara del ajuste fino. En el cuaderno 04 enseñamos al modelo a **responder**;
aquí solo le damos **versos** (sin preguntas) para que imite el estilo.

Entrenamos unos minutos con el texto de 2–3 obras y pedimos al modelo que continúe un parlamento.
Compara el resultado con el del modelo sin ajustar: ¿cambia el **estilo** (verso, léxico, ritmo)? ¿Sabe más **hechos**?

⏱️ Con GPU T4: unos 5 min. En CPU pon `"max_steps": 20` en los ajustes.

In [ ]:
#@title ⚙️ Preparación: ejecuta esta celda primero (no hace falta leer el código) { display-mode: "form" }
# ⚙️ Preparación: ejecuta esta celda primero (Mayús + Intro). Puedes repetirla sin problema.
import os, sys, importlib.util
REPO_URL = "https://github.com/DrKroll/llm-humanidades-salamanca"
REPO_DIR = "/content/llm-humanidades-salamanca"

if "google.colab" in sys.modules:
    # En Colab: descargamos el repositorio (solo la primera vez) y entramos en su carpeta
    if not os.path.exists(REPO_DIR):
        !git clone -q {REPO_URL} {REPO_DIR}
    os.chdir(REPO_DIR)
    # Colab ya trae torch, transformers, peft, etc.: solo instalamos lo que falte,
    # sin tocar las versiones preinstaladas
    paquetes = {"transformers": "transformers", "tokenizers": "tokenizers", "peft": "peft",
                "accelerate": "accelerate", "yaml": "pyyaml", "networkx": "networkx", "tqdm": "tqdm"}
    faltan = [pip for mod, pip in paquetes.items() if importlib.util.find_spec(mod) is None]
    if faltan:
        !pip install -q {" ".join(faltan)}
else:
    # En tu ordenador: subimos de carpeta hasta encontrar config.yaml
    while not os.path.exists("config.yaml") and os.getcwd() != "/":
        os.chdir("..")
# El código del taller (carpeta src/dhllm) se importa directamente, sin reiniciar
if os.path.abspath("src") not in sys.path:
    sys.path.insert(0, os.path.abspath("src"))
print("Carpeta de trabajo:", os.getcwd())

## ⚙️ Ajustes

In [ ]:
from dhllm import load_config, set_seed
cfg = load_config(overrides={
    "calderon_estilo": {
        # "max_steps": 20,   # ← quita el «#» en CPU
    }
})
ajustes = cfg.calderon_estilo
set_seed(cfg.seed)
INICIO = "Segismundo:\n    ¡Ay mísero de mí, ay infelice!"

## 1. Textos y modelo

In [ ]:
from dhllm import calderon as cal
from dhllm.finetune import load_base_model, make_chunks, prepare_model, finetune_on_texts, generate_hf
obras = cal.load_plays(cfg.calderon.tei_folder)
textos_estilo = [cal.play_to_text(o) for o in obras[: ajustes.n_plays]]
modelo, tokenizer = load_base_model(cfg.calderon.base_model)
print("ANTES:\n", generate_hf(modelo, tokenizer, INICIO, max_new_tokens=80))

## 2. Entrenar y comparar

In [ ]:
tr, va = make_chunks(textos_estilo, tokenizer, block_size=ajustes.block_size)
modelo = prepare_model(modelo, "lora", ajustes.lora_r, ajustes.lora_alpha)
finetune_on_texts(modelo, tr, va, ajustes)
print("DESPUÉS:\n", generate_hf(modelo, tokenizer, INICIO, max_new_tokens=80))

## 🖐️ Tu turno
1. Cambia `INICIO` por el comienzo de otro parlamento o por una frase en prosa.
2. Usa solo los autos sacramentales (`obras[8:11]`). ¿Cambia el estilo?
3. **Para pensar:** el modelo «escribe como Calderón», pero ¿podríamos atribuirle esos versos? ¿Qué diferencia hay con la estilometría?